# Agent + Knowledge Base

Queries `health-banking-kb` using **two calls**:
1. `KnowledgeBaseRetrievalClient.retrieve()` — extractive mode, returns raw chunks (no LLM call)
2. `openai_client.chat.completions` — synthesizes the answer from those chunks

**Run order:** Setup → Clients → Chat

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

search_url            = os.getenv("SEARCH_SERVICE_URL")
search_api_key        = os.getenv("SEARCH_SERVICE_API_KEY")
foundry_endpoint      = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
foundry_model_api_key = os.getenv("FOUNDRY_MODEL_API_KEY")
llm_model_name        = os.getenv("LLM_MODEL_NAME")

KB_NAME = "health-banking-kb"

print("search_url      :", search_url)
print("foundry_endpoint:", foundry_endpoint)
print("llm_model_name  :", llm_model_name)

search_url      : https://searchservice-nagh.search.windows.net
foundry_endpoint: https://foundry-rag-nagh.services.ai.azure.com/api/projects/rag
llm_model_name  : gpt-4.1


## Step 1 — Set up clients

In [7]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from azure.ai.projects import AIProjectClient

# Retrieval client — hits the AI Search KB endpoint directly (no LLM call)
retrieval_client = KnowledgeBaseRetrievalClient(
    endpoint=search_url,
    credential=AzureKeyCredential(search_api_key),
    knowledge_base_name=KB_NAME,
)

from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(
    endpoint=foundry_endpoint,
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()

print("Clients ready.")

Clients ready.


## Step 2 — Chat

`ask()` does two things:
- Calls `KnowledgeBaseRetrievalClient.retrieve()` → raw document chunks (no LLM)
- Calls `openai_client.chat.completions` → synthesizes an answer from those chunks

Pass a question to start. Call `ask()` again to continue the conversation (history is kept).

In [13]:
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalSemanticIntent,
    KnowledgeRetrievalMinimalReasoningEffort,
)

# Accumulates turns for your own LLM synthesis context
chat_history = []


def ask(question: str) -> str:

    # =========================================================
    # STEP A — Retrieve relevant chunks from Foundry IQ KB
    # =========================================================

    retrieval_request = KnowledgeBaseRetrievalRequest(
        intents=[
            KnowledgeRetrievalSemanticIntent(
                search=question
            )
        ],
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
        output_mode="extractiveData",
        max_output_documents=8,
    )

    retrieval_response = retrieval_client.retrieve(
        retrieval_request
    )

    # =========================================================
    # Extract retrieved document chunks
    # =========================================================

    chunks = []

    for msg in retrieval_response.response:
        for block in msg.content:
            if hasattr(block, "text") and block.text:
                chunks.append(block.text)

    context = (
        "\n\n---\n\n".join(chunks)
        if chunks
        else "No relevant documents found."
    )

    # =========================================================
    # Print sources returned by the Knowledge Base
    # =========================================================

    if retrieval_response.references:
        print("\nSources:")

        for ref in retrieval_response.references:
            print(
                getattr(
                    ref,
                    "source_name",
                    str(ref)
                )
            )

    # =========================================================
    # STEP B — Synthesize answer using your LLM
    # =========================================================

    synthesis = openai_client.chat.completions.create(
        model=llm_model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a financial and health economics "
                    "research assistant. "

                    "Answer ONLY using the provided documents. "

                    "Do not use your general knowledge. "

                    "Cite the source table or document name when "
                    "referencing data. "

                    "If the answer is not contained in the "
                    "provided documents, say: 'I don't know.'"
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Documents:\n\n"
                    f"{context}\n\n"
                    f"Question: {question}"
                ),
            },
        ],
    )

    answer = synthesis.choices[0].message.content

    # =========================================================
    # Save conversation history
    # =========================================================

    chat_history.append(
        {
            "role": "user",
            "content": question,
        }
    )

    chat_history.append(
        {
            "role": "assistant",
            "content": answer,
        }
    )

    return answer


# =============================================================
# TEST
# =============================================================

answer = ask(
    "Which banks are headquartered in Texas?"
)

print("\nAnswer:")
print(answer)


Sources:
{'type': 'indexedSql', 'id': '0', 'activitySource': 2, 'rerankerScore': 2.406527, 'sourceData': None}
{'type': 'indexedSql', 'id': '1', 'activitySource': 1, 'rerankerScore': 2.0186484, 'sourceData': None}
{'type': 'indexedSql', 'id': '2', 'activitySource': 2, 'rerankerScore': 1.9180169, 'sourceData': None}
{'type': 'indexedSql', 'id': '3', 'activitySource': 2, 'rerankerScore': 1.9174861, 'sourceData': None}
{'type': 'indexedSql', 'id': '4', 'activitySource': 2, 'rerankerScore': 1.8134892, 'sourceData': None}
{'type': 'indexedSql', 'id': '5', 'activitySource': 2, 'rerankerScore': 1.8132386, 'sourceData': None}
{'type': 'indexedSql', 'id': '6', 'activitySource': 2, 'rerankerScore': 1.8081868, 'sourceData': None}
{'type': 'indexedSql', 'id': '7', 'activitySource': 2, 'rerankerScore': 1.8036555, 'sourceData': None}

Answer:
All the banks listed are headquartered in Texas:

- Big Bend Banks, N.A. (Presidio, TX 79845)
- Transpecos Banks, SSB (TX)
- First National Bank Texas (Lubboc

In [11]:
# Multi-source follow-up — pulls from fedfunds + mortgage knowledge sources
followup = ask("What is the spread between the federal funds rate and the 30-year mortgage rate?")
print(followup)


Sources:
{'type': 'azureBlob', 'id': '0', 'activitySource': 0, 'rerankerScore': 1.4663088, 'blobUrl': 'https://safoundryrag.blob.core.windows.net/docs-pdf/006.pdf', 'sourceData': None}
I don't know.
